# تحلیل نتایج و ممیزی

این دفترچه گزارش‌های ممیزی اصلاح‌شده را می‌خواند، معیارها را مستقیماً از ردیف‌های نتیجه دوباره محاسبه و با خلاصهٔ CSV مقایسه می‌کند، سپس عملکرد را برحسب نوع شرط و ستون منبع ترسیم می‌کند. فایل‌های ورودی باید در `data/` باشند.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path('data')
RESULTS = DATA_DIR / 'evaluation_results_corrected.csv'
SUMMARY = DATA_DIR / 'evaluation_summary_corrected.csv'
for path in (RESULTS,SUMMARY):
    if not path.exists(): raise FileNotFoundError(f'{path} not found')
records = pd.read_csv(RESULTS)
summary = pd.read_csv(SUMMARY)
print('Audit records:',records.shape,'| summary rows:',summary.shape[0])
display(records.head())
display(summary.head())

## بازحساب معیارها و تطبیق با خلاصهٔ ثبت‌شده

In [ ]:
def calculate_metrics(frame):
    y=frame.ground_truth.astype(str).str.strip().str.lower().eq('deceptive')
    p=frame.prediction.astype(str).str.strip().str.lower().eq('deceptive')
    tp=int((y&p).sum()); fp=int((~y&p).sum()); fn=int((y&~p).sum()); tn=int((~y&~p).sum())
    precision=tp/(tp+fp) if tp+fp else 0.0
    recall=tp/(tp+fn) if tp+fn else 0.0
    f1=2*precision*recall/(precision+recall) if precision+recall else 0.0
    return {'n':len(frame),'correct':int((y==p).sum()),'accuracy':float((y==p).mean()) if len(frame) else 0.0,
            'TP':tp,'FP':fp,'FN':fn,'TN':tn,'precision':precision,'recall':recall,'f1':f1}

overall = calculate_metrics(records)
print('Recomputed overall:', overall)
print('Stored overall row:')
display(summary[summary.breakdown.eq('overall')])
assert overall['n']==int(summary.loc[summary.breakdown.eq('overall'),'n'].iloc[0])
assert overall['correct']==int(summary.loc[summary.breakdown.eq('overall'),'correct'].iloc[0])
assert abs(overall['accuracy']-float(summary.loc[summary.breakdown.eq('overall'),'accuracy'].iloc[0])) < 1e-10
print('Overall n/correct/accuracy match the stored summary.')

## عملکرد براساس شرط و ستون منبع
گروه‌های «شرط» عبارت‌اند از `label_type`، و تفکیک منبع از `source_column` استفاده می‌کند. هر دو نمودار با دادهٔ خام ممیزی محاسبه می‌شوند.

In [ ]:
def grouped_metrics(column):
    rows=[]
    for condition, group in records.groupby(column,dropna=False):
        rows.append({'condition':str(condition),**calculate_metrics(group)})
    return pd.DataFrame(rows).sort_values('condition')
by_condition=grouped_metrics('label_type')
by_source=grouped_metrics('source_column')
display(by_condition); display(by_source)
fig, axes = plt.subplots(1,2,figsize=(13,4.8))
for ax, data, title in [(axes[0],by_condition,'Accuracy by condition (label type)'),(axes[1],by_source,'Accuracy by source column')]:
    ax.bar(data.condition,data.accuracy*100,color='#3977a8')
    ax.set_title(title); ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0,100); ax.tick_params(axis='x',rotation=30)
    ax.grid(axis='y',alpha=.25)
    for i,v in enumerate(data.accuracy*100): ax.text(i,v+1,f'{v:.1f}%',ha='center',fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Sensitivity check: corrected audit should have 1,152 records and four equally sized sources.
assert len(records)==1152
assert records.source_column.value_counts().nunique()==1
print('Verified source counts:', records.source_column.value_counts().to_dict())
print('Key source accuracies (%):', (by_source.set_index('condition').accuracy*100).round(2).to_dict())